# Granger Causality: Uncovers potential causal relationships between genes over time. 

Pre-requisites:
- Trained MIOFlow trajectories

In this notebook:
- Perform Granger causality testing:
    - Between each pair of genes (or a subset of input/output gene groups).

    - Using lag-1 autoregressive models.

- Output a p-value matrix:

    - Rows: Response genes

    - Columns: Predictor genes

    - Values: Significance of Granger-causal influence

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
from statsmodels.tsa.stattools import grangercausalitytests
import scanpy as sc
from joblib import Parallel, delayed
import os

lag_order = 1 # since we aggregated the data in to 9 bins we only need 1 lag
maxlag = (
    lag_order,  # becuase we got this value before. We are not suppose to add 1 to it
)

RAW_DATA_DIR = os.path.join('../../data', 'raw')
PROCESSED_DATA_DIR = os.path.join('../../data', 'processed')
RESULTS_DIR = os.path.join('../../results')

data_name = 'scRNAseq'
SAVE_PATH = os.path.join(RESULTS_DIR, data_name)

adata = sc.read(os.path.join(PROCESSED_DATA_DIR, 'adata_mioflow.h5ad'))
gene_names = adata.var_names

traj = np.load(os.path.join(SAVE_PATH,'trajectories_gene_space.npy'))

test = "ssr_chi2test"


In [8]:
def grangers_causation_matrix(
    data, in_variables, out_variables, test="ssr_chi2test", n_jobs=1, warn=False
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """

    def get_pval(dd):
        if warn:
            test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=True)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=FutureWarning)
                test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=False)
                # according to the documentation https://www.statsmodels.org/dev/generated/statsmodels.tsa.stattools.grangercausalitytests.html,
                # the dd has 2 columns, second causes the first.

        p_values = [test_result[i][0][test][1] for i in maxlag] # test_result[i][1] is the unrestricted model, test_result[i][1][0] is the restricted model
        coefs = [test_result[i][1][1].params[1] for i in maxlag] # x1, x2, const

        arg_min_p_value = np.argmin(p_values)
        min_p_value = p_values[arg_min_p_value]
        min_coef = coefs[arg_min_p_value]
        return (min_p_value, min_coef)

    out = Parallel(n_jobs=n_jobs)(
        delayed(get_pval)(data[[c, r]]) # this means r causes c, so r is be in and c is out
        for c in tqdm(out_variables, desc="Processing columns")  # Outer loop progress bar
        for r in in_variables  # Inner loop without progress bar
    )
    out_p = [p for (p,c) in out]
    out_c = [c for (p,c) in out]
    df_p = pd.DataFrame(
        np.array(out_p).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T # used the correct reshaping, and then transposed the matrix so the x and y are semantically correct (x causes y).
    df_c = pd.DataFrame(
        np.array(out_c).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T
    df_p.index = [var + "_x" for var in in_variables]
    df_p.columns = [var + "_y" for var in out_variables]
    df_c.index = [var + "_x" for var in in_variables]
    df_c.columns = [var + "_y" for var in out_variables]
    return df_p, df_c

def do_granger(trajs, in_genes, out_genes, n_jobs=1, warn=False):
    # in causes out
    trajs = trajs.T[::10]
    trajs = trajs - trajs.shift(1)
    trajs = trajs.dropna()
    out_traj_p, out_traj_c = grangers_causation_matrix(
        trajs, in_variables=in_genes, out_variables=out_genes, n_jobs=n_jobs, warn=warn
    )
    return out_traj_p, out_traj_c

Load the gene expression trajectories generated by MIOFlow along with the original dataset. For simplicity, we focus on the top 2,000 most highly variable genes.

In [9]:
trajectories = np.load('../../results/scRNAseq/trajectories_gene_space.npy', allow_pickle=True)
adata = sc.read('../../data/processed/adata_mioflow.h5ad')
genes = adata.var_names.to_list()
sc.pp.highly_variable_genes(adata, n_top_genes=250)
hv_mask = adata.var['highly_variable']
genes_hvg = hv_mask[hv_mask].index.tolist()
col_genes = np.array(genes_hvg)

Intersect the selected genes with the list of human transcription factors downloaded from humantfs.ccbr.utoronto.ca to identify the input genes that are transcription factors.

In [10]:
db_extract = pd.read_csv(os.path.join(RAW_DATA_DIR,'DatabaseExtract_v_1.01.csv'))
db_ensembl_ids = set(db_extract['Ensembl ID'].astype(str))
genes_ensembl_ids = set([g.split('(')[-1].replace(')', '').strip() for g in genes_hvg])
in_genes_ensembl = genes_ensembl_ids & db_ensembl_ids
in_genes = [g for g in genes_hvg if g.split('(')[-1].replace(')', '').strip() in in_genes_ensembl]

In [11]:
trajectories_hvg = trajectories[:, :, hv_mask.values]
avg_traj = trajectories_hvg.mean(axis=1)
trajectories_df = pd.DataFrame(avg_traj, columns=col_genes)
out_traj_p, out_traj_c = do_granger(trajectories_df.T, in_genes, genes_hvg, n_jobs=1, warn=False)

Processing columns: 100%|██████████| 250/250 [00:04<00:00, 59.59it/s]


Output of Granger Causality:

1. out_traj_p: p-value (smaller the better)
2. out_traj_c: coefficient (the sign indicates if it is up or down regulation)

In [12]:
out_traj_p

,AC007325.4 (ENSG00000278817)_y,ACTA1 (ENSG00000143632)_y,AFP (ENSG00000081051)_y,AGT (ENSG00000135744)_y,AHSG (ENSG00000145192)_y,ALDH1A1 (ENSG00000165092)_y,ALX1 (ENSG00000180318)_y,ANKRD37 (ENSG00000186352)_y,APOM (ENSG00000204444)_y,AREG (ENSG00000109321)_y,...,TSPAN8 (ENSG00000127324)_y,TWIST2 (ENSG00000233608)_y,TYRP1 (ENSG00000107165)_y,VTN (ENSG00000109072)_y,VWF (ENSG00000110799)_y,WFDC1 (ENSG00000103175)_y,WIF1 (ENSG00000156076)_y,XAGE2 (ENSG00000155622)_y,ZFP36 (ENSG00000128016)_y,ZNF592 (ENSG00000166716)_y
AGT (ENSG00000135744)_x,7.529554e-01,2.593584e-01,2.700382e-01,1.000000e+00,3.869066e-01,7.051492e-23,6.702391e-16,5.089031e-07,1.451076e-02,9.268398e-14,...,8.921719e-06,7.150463e-01,0.063825,1.094207e-01,6.247391e-30,7.559306e-06,4.247617e-01,1.021304e-09,0.307585,1.542893e-01
ALX1 (ENSG00000180318)_x,1.903788e-01,3.256805e-01,1.331254e-20,8.205585e-02,3.245520e-02,1.105739e-09,1.000000e+00,2.474670e-02,6.753030e-02,8.064317e-02,...,1.352994e-03,8.149039e-01,0.000270,8.211358e-01,1.942329e-11,6.175425e-10,1.701521e-01,4.027584e-03,0.028459,5.508520e-08
ASCL1 (ENSG00000139352)_x,7.089421e-01,2.664685e-03,2.591710e-02,3.094496e-01,1.998571e-01,8.421985e-08,3.644557e-07,9.585608e-05,1.222963e-01,1.816117e-01,...,1.092569e-04,9.529941e-01,0.039930,6.513366e-01,3.364674e-09,4.998128e-10,3.530545e-01,2.226526e-02,0.034937,1.480998e-01
ATOH1 (ENSG00000172238)_x,8.252961e-01,4.704123e-01,6.579191e-01,9.087874e-01,8.185549e-01,2.043505e-01,6.598964e-01,3.982745e-01,9.247654e-01,9.943263e-01,...,6.886101e-01,2.927325e-01,0.305737,2.456089e-01,2.428565e-01,1.304009e-01,7.819530e-01,6.395882e-01,0.603545,1.058341e-01
CDX4 (ENSG00000131264)_x,4.790756e-01,7.364916e-01,8.480978e-01,8.440124e-03,3.507629e-06,2.914774e-07,1.498933e-19,4.017934e-04,2.802314e-01,1.500751e-01,...,4.633535e-03,8.700017e-01,0.345959,1.598455e-01,8.242377e-12,3.877878e-04,2.872459e-01,1.193686e-01,0.721634,1.595732e-02
CYP1B1 (ENSG00000138061)_x,4.180900e-01,9.021615e-01,8.142887e-01,1.689671e-02,7.340812e-04,1.042695e-10,1.814488e-13,2.074980e-03,1.668787e-01,1.139292e-01,...,3.026079e-01,8.023554e-01,0.532201,2.627559e-01,3.030218e-17,3.296115e-03,2.075920e-01,1.477792e-02,0.910815,5.904482e-03
DLX5 (ENSG00000105880)_x,4.020481e-01,7.011252e-01,2.994453e-06,1.001625e-01,3.719348e-02,2.671131e-05,7.372750e-05,1.635010e-01,1.315649e-01,1.352511e-01,...,3.897602e-02,9.165999e-01,0.044613,7.435021e-01,1.007860e-05,1.621045e-03,2.130846e-01,1.257253e-02,0.234518,4.090563e-04
EDN1 (ENSG00000078401)_x,1.110474e-01,5.694423e-01,6.918696e-01,4.334991e-52,9.763569e-14,2.304370e-03,3.657691e-18,3.320121e-02,2.606645e-09,6.897784e-19,...,4.162035e-02,1.497300e-01,0.726998,5.009396e-03,2.264938e-04,3.904817e-12,4.012436e-02,8.373168e-05,0.768118,8.820373e-05
FAM200B (ENSG00000237765)_x,1.002863e-02,3.355303e-02,1.197029e-01,7.325639e-01,5.224453e-01,2.082997e-07,7.139541e-13,1.046243e-03,4.370457e-01,5.398055e-01,...,1.529533e-03,1.541441e-01,0.083271,1.542578e-01,7.766025e-08,1.744271e-24,1.036844e-02,1.257075e-01,0.252155,4.414765e-04
GSX2 (ENSG00000180613)_x,6.259886e-03,3.749027e-01,4.767109e-01,5.020240e-01,8.298728e-01,1.238728e-04,6.520313e-08,2.835768e-02,8.002402e-01,7.581047e-01,...,1.577307e-02,1.354506e-03,0.318725,2.792549e-02,6.553942e-05,3.192684e-17,1.749493e-03,4.953159e-01,0.551202,1.046644e-06


Save the p-values and coefficients to use for RITINI

In [13]:
out_traj_p.to_csv('../../data/processed/out_traj_p_250.csv')
out_traj_c.to_csv('../../data/processed/out_traj_c_250.csv')